# 04 — Agentic Loop Demo

End-to-end demo of the agentic iteration loop.  We invoke `iterate_from_run`
with `depth=1, dry_run=True`, inspect the generated proposal manifests,
and walk through what each follow-up spec would do if executed.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

repo_root = Path("__file__").parent.parent
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

## Pick a seed run

The agentic loop takes a completed run as input and proposes follow-up
experiments.  We grab the most recent succeeded run of any family.

In [ ]:
from mech_interp.storage.sqlite_store import SQLiteResultStore
from mech_interp.types import RunStatus

store = SQLiteResultStore(
    database_path=repo_root / "artifacts" / "results.db",
    artifact_dir=repo_root / "artifacts",
)

runs = store.list_runs(limit=50)
succeeded = [r for r in runs if r.status == RunStatus.SUCCEEDED]

if not succeeded:
    raise RuntimeError(
        "No succeeded runs found. Run any experiment first, e.g.: "
        "mech run --name acdc_lite"
    )

seed_run = succeeded[0]
seed_result = store.get_result(seed_run.id)
print(f"Seed run: {seed_run.id} | family={seed_run.family} | spec={seed_run.spec_name}")
print("Metrics:", json.dumps(seed_result.metrics if seed_result else {}, indent=2))

## Invoke `iterate_from_run` in dry-run mode

`dry_run=True` generates the proposal manifests without enqueuing them.
This lets us inspect what the loop *would* do without committing resources.

In [ ]:
from mech_interp.orchestration.iterate_loop import iterate_from_run

proposals = iterate_from_run(
    run=seed_run,
    result=seed_result,
    store=store,
    depth=1,
    dry_run=True,
)

print(f"Generated {len(proposals)} proposals at depth=1")

## Proposal manifest table

Each proposal is an `ExperimentSpec`-like dict with a name, family,
hypothesis, and parameter overrides.  The table below shows the key fields.

In [ ]:
rows = []
for p in proposals:
    if hasattr(p, "model_dump"):
        d = p.model_dump()
    elif hasattr(p, "__dict__"):
        d = p.__dict__
    else:
        d = dict(p)  # type: ignore[call-overload]
    rows.append({
        "name": d.get("name", "?"),
        "family": d.get("family", "?"),
        "hypothesis": str(d.get("hypothesis", ""))[:80],
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

## Walk through each follow-up spec

For each proposal we print its full parameter block so it is easy to see
exactly what would differ from the seed experiment.

In [ ]:
for i, p in enumerate(proposals, 1):
    if hasattr(p, "model_dump"):
        d = p.model_dump()
    elif hasattr(p, "__dict__"):
        d = p.__dict__
    else:
        d = dict(p)  # type: ignore[call-overload]
    print(f"--- Proposal {i}: {d.get('name', '?')} ---")
    params = d.get("parameters", {})
    print(json.dumps(params, indent=2, default=str)[:600])
    print()

## How the loop works

The iteration loop is a lightweight agentic controller:

1. **Inspect** the seed result's metrics and artifacts.
2. **Generate proposals** using `ProposalGenerator` subclasses registered
   per family (e.g. `ACDCEdgeProposalGenerator` widens the layer range or
   lowers τ if faithfulness is high).
3. **Enqueue** (or in dry-run mode, return) the proposals as new `ExperimentSpec`s.
4. **Recurse** up to `depth` levels once each child run completes.

The loop intentionally avoids model downloads — all proposals are parameter
variations on already-cached model weights.  Resource limits (CPU time, disk)
are enforced by `ResourcePolicy` before any enqueue.